In [43]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [44]:
# Load ratings
ratings = pd.read_csv(
    "/content/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

# Load movie titles
movies = pd.read_csv(
    "/content/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

movies = movies[[0, 1]]
movies.columns = ["movie_id", "title"]

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [45]:
user_item_matrix = ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
).fillna(0)

user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
user_similarity = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


In [47]:
def recommend_movies(user_id, top_n=5):

    # Get similarity scores for user
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]

    # Get movies user already watched
    user_movies = user_item_matrix.loc[user_id]
    watched_movies = user_movies[user_movies > 0].index

    # Weighted average rating
    weighted_ratings = pd.Series(dtype=float)

    for similar_user, similarity in similar_users.items():
        similar_user_ratings = user_item_matrix.loc[similar_user]
        weighted_ratings = weighted_ratings.add(
            similar_user_ratings * similarity,
            fill_value=0
        )

    # Remove already watched
    weighted_ratings = weighted_ratings.drop(watched_movies)

    # Top N recommendations
    recommendations = weighted_ratings.sort_values(ascending=False).head(top_n)

    # Get movie titles
    recommended_movies = movies[movies["movie_id"].isin(recommendations.index)]

    return recommended_movies["title"]

In [48]:
recommend_movies(user_id=5, top_n=5)

,title
55,Pulp Fiction (1994)
95,Terminator 2: Judgment Day (1991)
183,Army of Darkness (1993)
194,"Terminator, The (1984)"
745,Real Genius (1985)


In [49]:
def precision_at_k(user_id, k=5):

    recommended = recommend_movies(user_id, k)

    user_data = ratings[ratings["user_id"] == user_id]
    liked_movies = user_data[user_data["rating"] >= 4]["movie_id"]

    recommended_ids = movies[movies["title"].isin(recommended)]["movie_id"]

    relevant = len(set(recommended_ids) & set(liked_movies))

    return relevant / k

precision_at_k(5, 5)

0.0

In [50]:
#Item-Based Collaborative Filtering
item_similarity = cosine_similarity(user_item_matrix.T)